# 03 — Test Module 4: Node-Neutralized Scoring

Test the GCN encoder-decoder standalone (without multi-pass).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = '/content/drive/MyDrive/Project_GraphML/ms-zerogad'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
import torch
from ms_zerogad.data.loader import load_graph_dataset
from ms_zerogad.data.preprocessing import sparse_to_torch_dense, feature_to_torch, normalize_adjacency
from ms_zerogad.modules.unification import GlobalUnification
from ms_zerogad.modules.scoring import NodeNeutralizedScoringModule, compute_anomaly_score
from ms_zerogad.evaluation.metrics import compute_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load + unify Cora

In [ ]:
A_sp, X_sp, y_np = load_graph_dataset('/content/drive/MyDrive/Project_GraphML/ms-zerogad/ms_zerogad/data/raw/Cora.mat')
A = sparse_to_torch_dense(A_sp).to(device)
X = feature_to_torch(X_sp, dense=True).to(device)
y = torch.from_numpy(y_np).long().to(device)

module1 = GlobalUnification(d_prime=8).to(device)
X_unified = module1(X, A)
A_norm = normalize_adjacency(A, add_self_loops=True)

print(f'X_unified: {X_unified.shape}')
print(f'A_norm: {A_norm.shape}')

## Forward pass (untrained)

In [ ]:
scoring = NodeNeutralizedScoringModule(
    d_input=8, d_hidden=64, d_latent=32,
    num_encoder_layers=3, num_decoder_layers=2,
).to(device)

n_params = sum(p.numel() for p in scoring.parameters())
print(f'Scoring module parameters: {n_params:,}')

X_gen, Z = scoring(X_unified, A_norm)

print(f'X_gen: {X_gen.shape}')
print(f'Z: {Z.shape}')

## Quick training (single pass)

Train just the scoring module on Cora to verify it learns.

In [ ]:
from ms_zerogad.training.losses import per_pass_loss

optimizer = torch.optim.Adam(scoring.parameters(), lr=1e-3, weight_decay=5e-4)

scoring.train()
for epoch in range(50):
    optimizer.zero_grad()
    X_gen, Z = scoring(X_unified, A_norm)
    loss = per_pass_loss(X_unified, X_gen, Z, alpha=2.0, beta=0.5)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(scoring.parameters(), 1.0)
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}: loss={loss.item():.4f}')

## Evaluate

In [ ]:
scoring.eval()
with torch.no_grad():
    X_gen, Z = scoring(X_unified, A_norm)
    scores = compute_anomaly_score(X_unified, X_gen)

metrics = compute_metrics(scores, y)
print(f'Single-pass (Module 4 only) on Cora:')
print(f'  AUROC: {metrics["auroc"]:.4f}')
print(f'  AUPRC: {metrics["auprc"]:.4f}')